In [1]:
# Install required packages in the environment
# Run these in your terminal or a Jupyter cell:
# conda activate PyEnv
# pip install torch==2.0.1 numpy==1.26.4 transformers==4.45.2 datasets==3.0.1 trl==0.11.4

import json
import random
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
import torch

# STEP 7: Load Tokenizer and Model (TinyLlama)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Chat version for better prompt handling
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Set padding token to EOS token if not defined
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Load model without quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",  # Auto-selects MPS, CPU, or GPU
    torch_dtype=torch.float16  # Use float16 for model weights (MPS-compatible)
)

# Verify device (MPS or CPU)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")
model.to(device)

# STEP 8: Reload JSONL as HuggingFace Dataset
try:
    dataset = load_dataset("json", data_files="dataset_train_line.jsonl")["train"]
    # Limit to 1,000 examples
    dataset = dataset.select(range(min(30, len(dataset))))
    print(f"Loaded dataset with {len(dataset)} examples (limited to 30)")
except Exception as e:
    print("Failed to load dataset:", str(e))
    exit(1)

# Formatting function for SFTTrainer
def format_dataset(example):
    # Extract messages list
    messages = example.get("messages", [])
    
    # Initialize variables for user and assistant content
    user_content = ""
    assistant_content = ""
    
    # Loop through messages to find user and assistant roles
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        if role == "user":
            user_content = content
        elif role == "assistant":
            assistant_content = content
    
    # Check if both user and assistant content exist
    if not user_content or not assistant_content:
        # Debug: Print sample record if formatting fails
        print("Missing user or assistant content in record:", example)
        return {"text": ""}  # Return empty text to skip malformed records
    
    # Format as a single text field
    return {
        "text": f"User: {user_content}\n\nAssistant: {assistant_content}"
    }

# Apply formatting with error handling
try:
    formatted_dataset = dataset.map(format_dataset)
    # Filter out empty text entries
    formatted_dataset = formatted_dataset.filter(lambda x: x["text"] != "")
    print(f"Formatted dataset with {len(formatted_dataset)} examples after filtering")
except Exception as e:
    print("Dataset formatting failed:", str(e))
    exit(1)

# STEP 9: Tokenize the dataset
def tokenize_function(example):
    try:
        # Tokenize the text field, ensuring it’s padded and truncated
        tokenized = tokenizer(
            example["text"],
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )
        # Convert tensors to lists for dataset compatibility
        return {
            "input_ids": tokenized["input_ids"].squeeze().tolist(),
            "attention_mask": tokenized["attention_mask"].squeeze().tolist()
        }
    except Exception as e:
        print("Tokenization error for example:", example)
        return {"input_ids": [], "attention_mask": []}  # Return empty to skip

# Tokenize the formatted dataset
tokenized_dataset = None
try:
    tokenized_dataset = formatted_dataset.map(
        tokenize_function,
        batched=False,  # Process one example at a time to isolate errors
        remove_columns=["text"]  # Remove text after tokenization
    )
    # Filter out examples with empty input_ids
    tokenized_dataset = tokenized_dataset.filter(lambda x: len(x["input_ids"]) > 0)
    print(f"Tokenized dataset with {len(tokenized_dataset)} examples")
except Exception as e:
    print("Tokenization failed:", str(e))
    print("Falling back to formatted_dataset for training")
    tokenized_dataset = formatted_dataset  # Fallback to raw text dataset

# STEP 10: Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./cbt_cot_model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    save_steps=200,
    logging_steps=20,
    learning_rate=2e-5,
    fp16=False,  # Disable fp16 for MPS compatibility
    bf16=True,   # Use bf16, supported on MPS
    optim="adamw_torch",
    save_strategy="steps",
    save_total_limit=2,
    logging_dir="./logs",
)

# STEP 11: Fine-Tune Using SFTTrainer
if tokenized_dataset is None:
    print("No dataset available for training")
    exit(1)

print(f"Training with dataset of size {len(tokenized_dataset)}")
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
)

trainer.train()
trainer.save_model()

# STEP 12: Inference
from transformers import pipeline

# Check NumPy availability
try:
    import numpy
    print(f"NumPy version: {numpy.__version__}")
except ImportError:
    print("Warning: NumPy is not available. Using pure PyTorch processing.")

pipe = pipeline("text-generation", model="./cbt_cot_model", tokenizer=tokenizer, device_map="auto")

# Create prompt (adjusted for TinyLlama chat format)
prompt = tokenizer.apply_chat_template(
    [
        {
            "role": "user",
            "content": "I can’t stop overthinking at night and it’s ruining my sleep. What should I do?"
        }
    ],
    tokenize=False,
    add_generation_prompt=True
)

# Generate response with custom handling to avoid NumPy
try:
    # Tokenize prompt manually
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    # Generate with model directly
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=400,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        num_return_sequences=1,
        pad_token_id=tokenizer.pad_token_id,
    )
    # Decode output
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
except Exception as e:
    print("Inference failed:", str(e))
    exit(1)

print(response)

Using device: mps


Generating train split: 0 examples [00:00, ? examples/s]

Loaded dataset with 30 examples (limited to 30)


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Filter:   0%|          | 0/30 [00:00<?, ? examples/s]

...

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

  0%|          | 0/6 [00:00<?, ?it/s]

{'train_runtime': 75.9032, 'train_samples_per_second': 0.79, 'train_steps_per_second': 0.079, 'train_loss': 159.44205729166666, 'num_tokens': 26624.0, 'mean_token_accuracy': 0.08369712416942303, 'epoch': 1.8}
NumPy version: 1.26.4


Device set to use mps
/opt/anaconda3/envs/PyEnv/lib/python3.9/site-packages/transformers/pytorch_utils.py:335: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


<|user|>
I can’t stop overthinking at night and it’s ruining my sleep. What should I do? 
<|assistant|>

